# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [5]:
"""A matrix class backed by a list of lists, plus standalone helper constructors."""

import numbers


# ---------------------------------------------------------------- validation
def _check_dim(value, name):
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(f"{name} must be an int, got {type(value).__name__}")
    if value < 1:
        raise ValueError(f"{name} must be a positive integer, got {value}")


def _validate_rows(rows):
    """Check that `rows` is a non-empty, rectangular list of non-empty lists.
    Returns (n, m)."""
    if not isinstance(rows, list):
        raise TypeError("values must be a list of lists")
    if len(rows) == 0:
        raise ValueError("matrix must have at least one row")
    for i, row in enumerate(rows):
        if not isinstance(row, list):
            raise TypeError(f"row {i} is not a list (got {type(row).__name__})")
    m = len(rows[0])
    if m == 0:
        raise ValueError("matrix must have at least one column")
    for i, row in enumerate(rows):
        if len(row) != m:
            raise ValueError(
                f"row {i} has {len(row)} columns, but row 0 has {m}; "
                "all rows must have the same length"
            )
    return len(rows), m


def _is_scalar(x):
    return isinstance(x, numbers.Number) and not isinstance(x, bool)


class _Row:
    """A view on one row of a matrix, so that M[i][j] reads and writes the
    matrix itself while keeping its shape fixed."""

    def __init__(self, matrix, i):
        self._matrix = matrix
        self._i = i

    def __getitem__(self, j):
        return self._matrix[self._i, j]

    def __setitem__(self, j, value):
        self._matrix[self._i, j] = value

    def __len__(self):
        return self._matrix.m

    def __iter__(self):
        return iter(self._matrix._data[self._i])

    def __eq__(self, other):
        return list(self) == (list(other) if isinstance(other, _Row) else other)

    def __repr__(self):
        return repr(self._matrix._data[self._i])


# -------------------------------------------------------------------- matrix
class matrix:
    """An n x m matrix.

    Construction:
        matrix(n, m)            -> n x m matrix of zeros
        matrix([[..], [..]])    -> matrix from a list of lists (must be rectangular)

    Indexing / slicing:
        M[i][j], M[i, j]        element (negative indices allowed)
        M[i1:i2, j1:j2]         sub-matrix (any mix of ints and slices; the
        M[i1:i2]                result is always a matrix)
        M[i] = [...]            replace row i

    Copying values into an existing matrix (same size required, else ValueError):
        M1[:] = M2   or   M1.assign(M2)       (M2: matrix or list of lists)
    A bare `M1 = M2` only rebinds the name in Python and cannot be overloaded.

    Operators:  2*M, M*2, M+N, M-N, M*N (matrix product, same as M@N), M==N
    """

    def __init__(self, n, m=None):
        if isinstance(n, list):
            if m is not None:
                raise TypeError("give either (n, m) or a list of lists, not both")
            self.n, self.m = _validate_rows(n)
            self._data = [list(row) for row in n]  # copy: no aliasing
        else:
            if m is None:
                raise TypeError("m is required when n is given as an int")
            _check_dim(n, "n")
            _check_dim(m, "m")
            self.n, self.m = n, m
            self._data = [[0] * m for _ in range(n)]

    # ------------------------------------------------------------ helpers
    @staticmethod
    def _norm(index, size, axis):
        if isinstance(index, bool) or not isinstance(index, int):
            raise TypeError(f"{axis} index must be an int, got {type(index).__name__}")
        if not -size <= index < size:
            raise IndexError(f"{axis} index {index} out of range for size {size}")
        return index % size

    def _norm_key(self, key):
        i, j = key
        return self._norm(i, self.n, "row"), self._norm(j, self.m, "column")

    def _require_matrix(self, other, op):
        if not isinstance(other, matrix):
            raise TypeError(f"{op} needs a matrix, got {type(other).__name__}")

    def _require_same_shape(self, other, op):
        self._require_matrix(other, op)
        if self.shape() != other.shape():
            raise ValueError(
                f"{op} needs matrices of the same size, got "
                f"{self.n}x{self.m} and {other.n}x{other.m}"
            )

    # ------------------------------------------------------- basic methods
    def shape(self):
        """Return the tuple (n, m)."""
        return (self.n, self.m)

    def to_list(self):
        """Return the matrix as a (new) list of lists."""
        return [list(row) for row in self._data]

    def transpose(self):
        """Return a new matrix that is the transpose of this one."""
        return matrix([[self._data[i][j] for i in range(self.n)] for j in range(self.m)])

    def row(self, n):
        """Return row n as a new 1 x m matrix."""
        i = self._norm(n, self.n, "row")
        return matrix([list(self._data[i])])

    def column(self, n):
        """Return column n as a new n x 1 matrix."""
        j = self._norm(n, self.m, "column")
        return matrix([[r[j]] for r in self._data])

    def block(self, n_0, n_1, m_0, m_1):
        """Return the sub-matrix of rows n_0..n_1-1 and columns m_0..m_1-1
        (half-open ranges, like Python slices; n indexes rows, m columns)."""
        for name, v in (("n_0", n_0), ("n_1", n_1), ("m_0", m_0), ("m_1", m_1)):
            if isinstance(v, bool) or not isinstance(v, int):
                raise TypeError(f"{name} must be an int")
        if not 0 <= n_0 < n_1 <= self.n:
            raise ValueError(f"need 0 <= n_0 < n_1 <= {self.n}, got {n_0}, {n_1}")
        if not 0 <= m_0 < m_1 <= self.m:
            raise ValueError(f"need 0 <= m_0 < m_1 <= {self.m}, got {m_0}, {m_1}")
        return matrix([r[m_0:m_1] for r in self._data[n_0:n_1]])

    # ------------------------------------------------------- indexing
    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i], M[i, j] or slices")
            ri, ci = key
        else:
            ri, ci = key, None  # None: "all columns" / row-proxy case

        if ci is None and not isinstance(ri, slice):
            return _Row(self, self._norm(ri, self.n, "row"))
        if ci is None:
            ci = slice(None)
        if not isinstance(ri, slice) and not isinstance(ci, slice):
            i, j = self._norm_key((ri, ci))
            return self._data[i][j]

        rows = range(*ri.indices(self.n)) if isinstance(ri, slice) \
            else [self._norm(ri, self.n, "row")]
        cols = range(*ci.indices(self.m)) if isinstance(ci, slice) \
            else [self._norm(ci, self.m, "column")]
        if len(rows) == 0 or len(cols) == 0:
            raise ValueError("slice selects an empty matrix")
        return matrix([[self._data[i][j] for j in cols] for i in rows])

    def __setitem__(self, key, value):
        if isinstance(key, slice):
            if key != slice(None):
                raise IndexError("only the full slice M[:] is supported")
            self.assign(value)
        elif isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i] or M[i, j]")
            i, j = self._norm_key(key)
            self._data[i][j] = value
        else:
            i = self._norm(key, self.n, "row")
            if not isinstance(value, (list, _Row)) or len(value) != self.m:
                raise ValueError(f"row must be a list with {self.m} entries")
            self._data[i] = list(value)

    def assign(self, other):
        """Copy the values of `other` (a matrix or list of lists) into self."""
        if isinstance(other, matrix):
            rows, shape = other._data, other.shape()
        else:
            shape = _validate_rows(other)  # raises if badly specified
            rows = other
        if shape != self.shape():
            raise ValueError(
                f"size mismatch: cannot assign {shape[0]}x{shape[1]} "
                f"to {self.n}x{self.m} matrix"
            )
        self._data = [list(row) for row in rows]  # self untouched on error

    # ------------------------------------------------------- arithmetic
    def scalarmul(self, c):
        """Return c * M."""
        if not _is_scalar(c):
            raise TypeError(f"scalar must be a number, got {type(c).__name__}")
        return matrix([[c * x for x in row] for row in self._data])

    def add(self, N):
        self._require_same_shape(N, "add")
        return matrix([[a + b for a, b in zip(r, s)]
                       for r, s in zip(self._data, N._data)])

    def sub(self, N):
        self._require_same_shape(N, "sub")
        return matrix([[a - b for a, b in zip(r, s)]
                       for r, s in zip(self._data, N._data)])

    def mat_mult(self, N):
        """Matrix product M N; requires M.m == N.n."""
        self._require_matrix(N, "mat_mult")
        if self.m != N.n:
            raise ValueError(
                f"mat_mult: inner sizes differ, got {self.n}x{self.m} times {N.n}x{N.m}"
            )
        return matrix([
            [sum(self._data[i][k] * N._data[k][j] for k in range(self.m))
             for j in range(N.m)]
            for i in range(self.n)
        ])

    def element_mult(self, N):
        """Element-wise (Hadamard) product."""
        self._require_same_shape(N, "element_mult")
        return matrix([[a * b for a, b in zip(r, s)]
                       for r, s in zip(self._data, N._data)])

    def equals(self, N):
        """True if N is a matrix of the same size with equal entries."""
        self._require_matrix(N, "equals")
        return self._data == N._data

    # ------------------------------------------------------- operators
    def __add__(self, other):
        return self.add(other) if isinstance(other, matrix) else NotImplemented

    def __sub__(self, other):
        return self.sub(other) if isinstance(other, matrix) else NotImplemented

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)
        if _is_scalar(other):
            return self.scalarmul(other)
        return NotImplemented

    def __rmul__(self, other):
        return self.scalarmul(other) if _is_scalar(other) else NotImplemented

    def __matmul__(self, other):
        return self.mat_mult(other) if isinstance(other, matrix) else NotImplemented

    def __neg__(self):
        return self.scalarmul(-1)

    def __eq__(self, other):
        return self.equals(other) if isinstance(other, matrix) else NotImplemented

    __hash__ = None  # mutable

    def __repr__(self):
        return f"matrix({self._data!r})"

    def __str__(self):
        return "\n".join(" ".join(str(x) for x in row) for row in self._data)


# ------------------------------------------------- special-matrix functions
def constant(n, m, c):
    """n x m matrix filled with the float c."""
    if not isinstance(c, (int, float)) or isinstance(c, bool):
        raise TypeError("c must be a real number")
    _check_dim(n, "n")
    _check_dim(m, "m")
    return matrix([[float(c)] * m for _ in range(n)])


def zeros(n, m):
    return constant(n, m, 0.0)


def ones(n, m):
    return constant(n, m, 1.0)


def eye(n):
    """n x n identity matrix (floats)."""
    _check_dim(n, "n")
    return matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])


In [8]:

A = matrix([[1, 2], [3, 4]])
B = matrix([[0, 1], [1, 0]])
C = matrix([[2, 0], [1, 3]])
I = eye(2)

print("1. (AB)C == A(BC):", (A * B) * C == A * (B * C))
print("   (AB)C =", ((A * B) * C).to_list())
print("   A(BC) =", (A * (B * C)).to_list())

print("2. A(B+C) == AB + AC:", A * (B + C) == A * B + A * C)
print("   A(B+C) =", (A * (B + C)).to_list())
print("   AB+AC  =", (A * B + A * C).to_list())

print("3. AB != BA:", A * B != B * A)
print("   AB =", (A * B).to_list())
print("   BA =", (B * A).to_list())

print("4. AI == A:", A * I == A)
print("   AI =", (A * I).to_list())

1. (AB)C == A(BC): True
   (AB)C = [[5, 3], [11, 9]]
   A(BC) = [[5, 3], [11, 9]]
2. A(B+C) == AB + AC: True
   A(B+C) = [[6, 7], [14, 15]]
   AB+AC  = [[6, 7], [14, 15]]
3. AB != BA: True
   AB = [[2, 1], [4, 3]]
   BA = [[3, 4], [1, 2]]
4. AI == A: True
   AI = [[1.0, 2.0], [3.0, 4.0]]
